In [12]:
from importlib.resources import files
from pathlib import Path
from neuromodes.io import fetch_map, fetch_surf
from neuromodes.eigen import EigenSolver
from nsbutils.plotting_pyvista import plot_surf_single, plot_surf_video
from nsbutils.utils import unmask

In [16]:
# Load data
mesh, medmask = fetch_surf(density="32k")
mesh_plot = {"v": mesh.vertices, "t": mesh.faces} # for plotting
map = fetch_map('myelinmap')  # 'fcgradient1'

nmodes = 1000

In [18]:
# Define external input
parc_file = files("neuromodes").parent / "docs" / "validation" / "data"/ "Q1-Q6_RelatedParcellation210.L.CorticalAreas_dil_Colors.32k_fs_LR.label.gii"
parc = nib.load(parc_file)

# Create mask for V1
label_to_key = {lab.label: lab.key for lab in parc.labeltable.labels}
V1_mask = parc.darrays[0].data == label_to_key['L_V1_ROI']

# Simulation parameters
dt = 0.1
nt = 250

# Create a 10 ms external input with amplitude 20.0 to V1
ext_input = np.zeros((mesh.vertices.shape[0], nt))
ext_input[V1_mask, 10:20] = 20.0
ext_input = ext_input[medmask, :]
print(ext_input.shape)

p = plot_surf_single(mesh_plot, data=unmask(ext_input[:, 10], medmask), cmap="viridis")
p.show()

(29696, 250)


Widget(value='<iframe src="http://localhost:61348/index.html?ui=P_0x1630ca980_4&reconnect=auto" class="pyvista…

In [19]:
solver = EigenSolver(mesh, mask=medmask, hetero=None, aniso=None)
solver.solve(n_modes=nmodes)
activity = solver.simulate_waves(ext_input=ext_input, nt=nt, dt=dt, gamma=0.3)

# Create video with improved rendering
video_file = plot_surf_video(
    surf=mesh_plot, 
    data_timeseries=activity,
    filename='cortex_iso_waves.mp4',
    framerate=50,
    cmap='seismic',
)

print(f"Video saved: {video_file}")

ValueError: `data_timeseries` shape (29696, 250) does not match mesh vertices (32492,).

In [ ]:
solver_hetero = EigenSolver(mesh, mask=None, hetero=map, aniso=None, alpha=1)
solver_hetero.solve(n_modes=nmodes)
activity_hetero = solver_hetero.simulate_waves(ext_input=ext_input, nt=nt, dt=dt, gamma=0.3)

# Create video with improved rendering
video_file = plot_surf_video(
    surf=mesh_plot, 
    data_timeseries=activity_hetero,
    filename='cortex_hetero_waves.mp4',
    framerate=50,
    cmap='seismic',
)

print(f"Video saved: {video_file}")

Widget(value='<iframe src="http://localhost:52678/index.html?ui=P_0x327eb7c10_4&reconnect=auto" class="pyvista…

Video saved: hetero_waves.mp4


In [14]:
# from lapy.diffgeo import tria_compute_gradient
# from lapy.plot import plot_tria_mesh

# grad3d = tria_compute_gradient(solver.geometry, central_patch)
# plot_tria_mesh(solver.geometry, tfunc=grad3d)

In [ ]:
solver_aniso = EigenSolver(mesh, mask=None, hetero=None, aniso=map, beta=10)
solver_aniso.solve(n_modes=nmodes)
activity_aniso = solver_aniso.simulate_waves(ext_input=ext_input, nt=nt, dt=dt, gamma=0.3)

# Create video with improved rendering
video_file = plot_surf_video(
    surf=mesh_plot, 
    data_timeseries=activity_aniso,
    filename='cortex_aniso_waves.mp4',
    framerate=50,
    cmap='seismic',
)

print(f"Video saved: {video_file}")

Widget(value='<iframe src="http://localhost:52678/index.html?ui=P_0x30748db40_4&reconnect=auto" class="pyvista…

Video saved: aniso_waves.mp4
